# LeetCode 1525 — Number of Good Ways to Split a String bằng Python

Link đề: https://leetcode.com/problems/number-of-good-ways-to-split-a-string/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Dynamic Programming, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def numSplits(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá "cắt chuỗi tại mọi vị trí" và nhận ra cần biết **số ký tự khác nhau của mọi tiền tố và mọi hậu tố**.
2. Viết được 4 cách: brute force `set` hai phía `O(n²)`, **mảng tiền tố/hậu tố số ký tự khác nhau** `O(n)`, **hai bộ đếm cập nhật dần** (`set` trái + `Counter` phải), và cách **lần xuất hiện đầu/cuối** của mỗi chữ cái.
3. Dùng `collections.Counter` đúng cách: xoá khoá khi đếm về 0 để `len()` phản ánh số ký tự khác nhau.
4. Biết biểu diễn tập chữ cái bằng **bitmask 26 bit** (`mask |= 1 << k`, `bin(mask).count("1")`).
5. Cẩn thận chỉ số khi ghép mảng tiền tố với mảng hậu tố (lệch 1 là lỗi phổ biến nhất).

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Một cách cắt là **tốt** nếu chia `s` thành hai chuỗi **khác rỗng** `sleft`, `sright` (`sleft + sright = s`) sao cho **số chữ cái khác nhau** trong `sleft` bằng trong `sright`. Trả về số cách cắt tốt.

**Ví dụ 1:** `s = "aacaba"` → `2`. Năm cách cắt: `("a","acaba")` 1 vs 3, `("aa","caba")` 1 vs 3, `("aac","aba")` 2 vs 2 ✔, `("aaca","ba")` 2 vs 2 ✔, `("aacab","a")` 3 vs 1.

**Ví dụ 2:** `s = "abcd"` → `1` (`("ab","cd")`).

**Ràng buộc:** `1 ≤ len(s) ≤ 10⁵`, `s` chỉ gồm chữ cái thường.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [("aacaba", 2), ("abcd", 1), ("aaaaa", 4), ("acbadbaada", 2)]
for s, want in examples:
    print(repr(s), "->", want)

'aacaba' -> 2
'abcd' -> 1
'aaaaa' -> 4
'acbadbaada' -> 2


Hai ví dụ cuối là ví dụ quen thuộc của bài (kiểm chứng bằng brute force ở mục 5; nên đối chiếu lại đề gốc). Cắt tại vị trí `i` (`1 ≤ i ≤ n−1`) nghĩa là `sleft = s[:i]`, `sright = s[i:]`.

In [2]:
s = "aacaba"
for i in range(1, len(s)):
    L, R = s[:i], s[i:]
    print(f"i={i}: {L!r:8} | {R!r:8} -> {len(set(L))} vs {len(set(R))} {'tốt' if len(set(L)) == len(set(R)) else ''}")

i=1: 'a'      | 'acaba'  -> 1 vs 3 
i=2: 'aa'     | 'caba'   -> 1 vs 3 
i=3: 'aac'    | 'aba'    -> 2 vs 2 tốt
i=4: 'aaca'   | 'ba'     -> 2 vs 2 tốt
i=5: 'aacab'  | 'a'      -> 3 vs 1 


## 3. Kiến thức nền

### 3.1 `set` và số phần tử khác nhau

`len(set(x))` = số phần tử khác nhau. Thêm phần tử: `st.add(c)` (`O(1)` trung bình).

In [3]:
print(len(set("aacaba")), set("aacaba"))
st = set()
for c in "abca":
    st.add(c)
    print(c, st, len(st))

3 {'b', 'a', 'c'}
a {'a'} 1
b {'a', 'b'} 2
c {'a', 'b', 'c'} 3
a {'a', 'b', 'c'} 3


### 3.2 `collections.Counter` — và bẫy khoá có giá trị 0

`Counter` đếm số lần xuất hiện. Giảm đếm về 0 **không tự xoá khoá** → `len(counter)` vẫn tính chữ đó. Phải `del cnt[c]` khi về 0.

In [4]:
cnt = Counter("aab")
print(cnt, len(cnt))
cnt["b"] -= 1
print("sau khi giảm b:", cnt, "len =", len(cnt), "<- vẫn 2!")
if cnt["b"] == 0:
    del cnt["b"]
print("sau khi xoá:", cnt, "len =", len(cnt))

Counter({'a': 2, 'b': 1}) 2
sau khi giảm b: Counter({'a': 2, 'b': 0}) len = 2 <- vẫn 2!
sau khi xoá: Counter({'a': 2}) len = 1


### 3.3 Mảng tiền tố / hậu tố

`pre[i]` = số chữ khác nhau trong `s[:i]` (độ dài `i`), `suf[i]` = trong `s[i:]`. Cắt tại `i` tốt ⇔ `pre[i] == suf[i]`.

In [5]:
s = "aacaba"
n = len(s)
pre = [0] * (n + 1)
seen = set()
for i, c in enumerate(s):
    seen.add(c)
    pre[i + 1] = len(seen)
suf = [0] * (n + 1)
seen = set()
for i in range(n - 1, -1, -1):
    seen.add(s[i])
    suf[i] = len(seen)
print("pre =", pre)
print("suf =", suf)

pre = [0, 1, 1, 2, 2, 3, 3]
suf = [3, 3, 3, 2, 2, 1, 0]


### 3.4 Bitmask cho tập chữ cái

Chữ thứ `k` (`a = 0`) ứng với bit `1 << k`. Hợp tập = `|`, số phần tử = số bit 1.

In [6]:
mask = 0
for c in "cab":
    mask |= 1 << (ord(c) - 97)
print(bin(mask), "số chữ khác nhau =", bin(mask).count("1"))

0b111 số chữ khác nhau = 3


### 3.5 Lần xuất hiện đầu / cuối

`first[c]`, `last[c]` — chỉ số đầu tiên / cuối cùng của chữ `c`. Chữ `c` có mặt trong `s[:i]` ⇔ `first[c] < i`; có mặt trong `s[i:]` ⇔ `last[c] ≥ i`.

In [7]:
s = "aacaba"
first, last = {}, {}
for i, c in enumerate(s):
    first.setdefault(c, i)
    last[c] = i
print("first =", first, "| last =", last)

first = {'a': 0, 'c': 2, 'b': 4} | last = {'a': 5, 'c': 2, 'b': 4}


## 4. Ý tưởng nền tảng / trực giác

Khi dời vị trí cắt sang phải một bước, **một** ký tự chuyển từ phải sang trái:
- Số chữ khác nhau bên trái **không giảm** (tăng 1 nếu ký tự đó mới xuất hiện lần đầu).
- Số chữ khác nhau bên phải **không tăng** (giảm 1 nếu đó là lần xuất hiện cuối).

Tính tay `s = "aacaba"`:

| i (cắt) | trái s[:i] | phải s[i:] | pre[i] | suf[i] | tốt? |
|---|---|---|---|---|---|
| 1 | a | acaba | 1 | 3 | |
| 2 | aa | caba | 1 | 3 | |
| 3 | aac | aba | 2 | 2 | ✔ |
| 4 | aaca | ba | 2 | 2 | ✔ |
| 5 | aacab | a | 3 | 1 | |

Vì `pre` không giảm và `suf` không tăng, các vị trí tốt tạo thành **một đoạn liên tiếp** (có thể rỗng) — một quan sát thú vị (mục 15).

In [8]:
s = "aacaba"
for i in range(1, len(s)):
    moved = s[i - 1]
    print(f"cắt tại {i}: ký tự chuyển sang trái = {moved!r}, pre={pre[i]}, suf={suf[i]}, tốt={pre[i] == suf[i]}")

cắt tại 1: ký tự chuyển sang trái = 'a', pre=1, suf=3, tốt=False
cắt tại 2: ký tự chuyển sang trái = 'a', pre=1, suf=3, tốt=False
cắt tại 3: ký tự chuyển sang trái = 'c', pre=2, suf=2, tốt=True
cắt tại 4: ký tự chuyển sang trái = 'a', pre=2, suf=2, tốt=True
cắt tại 5: ký tự chuyển sang trái = 'b', pre=3, suf=1, tốt=False


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi vị trí cắt `i ∈ [1, n−1]`, tạo `set(s[:i])` và `set(s[i:])`, so sánh kích thước.

In [9]:
def brute(s):
    n = len(s)
    cnt = 0
    for i in range(1, n):
        if len(set(s[:i])) == len(set(s[i:])):
            cnt += 1
    return cnt

for s, want in examples:
    print(repr(s), brute(s), "kỳ vọng", want)

'aacaba' 2 kỳ vọng 2
'abcd' 1 kỳ vọng 1
'aaaaa' 4 kỳ vọng 4
'acbadbaada' 2 kỳ vọng 2


**Độ phức tạp:** mỗi vị trí cắt tốn `O(n)` để cắt chuỗi và tạo set → Time `O(n²)` (`10¹⁰` với `n = 10⁵` — quá chậm), Space `O(n)`.

**Nút thắt:** hai vị trí cắt liền nhau chỉ khác nhau **một ký tự**, nhưng ta dựng lại set từ đầu. Hãy **tái sử dụng** thông tin: tiền tố/hậu tố hoặc cập nhật dần.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiền tố/hậu tố):** tính trước `pre[i]` và `suf[i]` cho mọi `i` trong hai lượt `O(n)`; đáp án = `#{i ∈ [1, n−1] : pre[i] == suf[i]}`.

**Insight 2 (cập nhật dần):** giữ `left` (set) và `right` (Counter, ban đầu là toàn chuỗi). Dời ký tự `c` sang trái: `left.add(c)`; `right[c] −= 1`, *nếu* về 0 *thì* xoá khoá. Sau mỗi bước so `len(left) == len(right)`.

**Insight 3 (đầu/cuối):** chỉ có tối đa 26 "sự kiện": `pre` tăng tại `first[c] + 1`, `suf` giảm sau `last[c]`. Đếm bằng mảng sai phân.

**Quy tắc:** *nếu* một đại lượng thay đổi đơn điệu khi dời điểm cắt *thì* có thể cập nhật `O(1)` mỗi bước thay vì tính lại.

## 7. Các cách tiếp theo

### Cách 2 — Mảng tiền tố / hậu tố số chữ khác nhau `O(n)`

1. `pre[i]` (độ dài tiền tố `i`) bằng một lượt trái → phải.
2. `suf[i]` bằng một lượt phải → trái.
3. Đếm `i` từ 1 đến `n − 1` có `pre[i] == suf[i]`.

In [10]:
def prefix_suffix(s):
    n = len(s)
    pre = [0] * (n + 1)
    seen = set()
    for i, c in enumerate(s):
        seen.add(c)
        pre[i + 1] = len(seen)
    suf = [0] * (n + 1)
    seen = set()
    for i in range(n - 1, -1, -1):
        seen.add(s[i])
        suf[i] = len(seen)
    return sum(1 for i in range(1, n) if pre[i] == suf[i])

for s, want in examples:
    print(repr(s), prefix_suffix(s), "kỳ vọng", want)

'aacaba' 2 kỳ vọng 2
'abcd' 1 kỳ vọng 1
'aaaaa' 4 kỳ vọng 4
'acbadbaada' 2 kỳ vọng 2


Bản nộp dùng bitmask thay cho set (nhanh, gọn — đúng tag *Bit Manipulation*):

In [11]:
class SolutionPrefix:
    def numSplits(self, s: str) -> int:
        n = len(s)
        pre = [0] * (n + 1)
        mask = 0
        for i, c in enumerate(s):
            mask |= 1 << (ord(c) - 97)
            pre[i + 1] = bin(mask).count("1")
        ans = 0
        mask = 0
        for i in range(n - 1, 0, -1):          # suf[i] = số chữ khác nhau của s[i:]
            mask |= 1 << (ord(s[i]) - 97)
            if bin(mask).count("1") == pre[i]:
                ans += 1
        return ans

print([SolutionPrefix().numSplits(s) for s, _ in examples])

[2, 1, 4, 2]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho mảng `pre`.

### Cách 3 — Hai bộ đếm cập nhật dần (set trái + Counter phải)

Đúng gợi ý của đề: "dùng hai HashMap đếm ký tự bên trái và bên phải".

In [12]:
def two_counters(s):
    left = set()
    right = Counter(s)
    ans = 0
    for c in s[:-1]:                 # chỉ cắt tới n-1 để hai phía khác rỗng
        left.add(c)
        right[c] -= 1
        if right[c] == 0:
            del right[c]
        if len(left) == len(right):
            ans += 1
    return ans

for s, want in examples:
    print(repr(s), two_counters(s), "kỳ vọng", want)

'aacaba' 2 kỳ vọng 2
'abcd' 1 kỳ vọng 1
'aaaaa' 4 kỳ vọng 4
'acbadbaada' 2 kỳ vọng 2


In [13]:
class SolutionCounter:
    def numSplits(self, s: str) -> int:
        right = Counter(s)
        left = set()
        ans = 0
        for c in s:
            left.add(c)
            right[c] -= 1
            if right[c] == 0:
                del right[c]
            ans += len(left) == len(right)
        return ans

print([SolutionCounter().numSplits(s) for s, _ in examples])

[2, 1, 4, 2]


Ở bản nộp ta duyệt cả ký tự cuối: khi đó bên phải rỗng (`len = 0`) còn bên trái `≥ 1` nên không bao giờ được tính — vẫn đúng mà gọn hơn.

**Độ phức tạp:** Time `O(n)`, Space `O(26) = O(1)`.

### Cách 4 — Lần xuất hiện đầu / cuối + mảng sai phân `O(n + 26)`

- `pre[i] = #{c : first[c] < i}` → mỗi chữ góp `+1` từ vị trí `first[c] + 1`.
- `suf[i] = #{c : last[c] ≥ i}` → mỗi chữ góp `+1` cho mọi `i ≤ last[c]`.

In [14]:
def first_last(s):
    n = len(s)
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    add_pre = [0] * (n + 2)
    cnt_last = [0] * (n + 1)
    for c in first:
        add_pre[first[c] + 1] += 1
        cnt_last[last[c]] += 1
    pre = [0] * (n + 1)
    run = 0
    for i in range(n + 1):
        run += add_pre[i]
        pre[i] = run
    suf = [0] * (n + 1)
    run = 0
    for i in range(n - 1, -1, -1):
        run += cnt_last[i]
        suf[i] = run
    return sum(1 for i in range(1, n) if pre[i] == suf[i])

for s, want in examples:
    print(repr(s), first_last(s), "kỳ vọng", want)

'aacaba' 2 kỳ vọng 2
'abcd' 1 kỳ vọng 1
'aaaaa' 4 kỳ vọng 4
'acbadbaada' 2 kỳ vọng 2


In [15]:
class SolutionFirstLast:
    def numSplits(self, s: str) -> int:
        n = len(s)
        first, last = {}, {}
        for i, c in enumerate(s):
            if c not in first:
                first[c] = i
            last[c] = i
        delta = [0] * (n + 1)            # delta[i] = thay đổi của (pre - suf) khi đi từ i-1 tới i
        for c in first:
            delta[first[c] + 1] += 1     # bên trái có thêm c từ vị trí cắt first[c]+1
            delta[last[c] + 1] += 1      # bên phải mất c từ vị trí cắt last[c]+1
        diff = -len(first)               # tại i = 0: pre = 0, suf = số chữ khác nhau
        ans = 0
        for i in range(1, n):
            diff += delta[i]
            ans += diff == 0
        return ans

print([SolutionFirstLast().numSplits(s) for s, _ in examples])

[2, 1, 4, 2]


Bản nộp gộp thành một đại lượng `diff = pre[i] − suf[i]`: bắt đầu `−(số chữ khác nhau)`, tăng 1 tại mỗi `first[c] + 1` (trái thêm chữ) và tại mỗi `last[c] + 1` (phải mất chữ). Cắt tốt ⇔ `diff == 0`.

**Độ phức tạp:** Time `O(n + 26)`, Space `O(n)` (có thể giảm còn `O(26)` bằng cách sắp xếp các sự kiện).

## 8. TRACE từng bước

Trace Cách 3 trên `"aacaba"`:

| bước | c | left | right (Counter) | len L | len R | tốt? |
|---|---|---|---|---|---|---|
| 1 | a | {a} | a:3, c:1, b:1 | 1 | 3 | |
| 2 | a | {a} | a:2, c:1, b:1 | 1 | 3 | |
| 3 | c | {a,c} | a:2, b:1 | 2 | 2 | ✔ |
| 4 | a | {a,c} | a:1, b:1 | 2 | 2 | ✔ |
| 5 | b | {a,b,c} | a:1 | 3 | 1 | |

In [16]:
def trace_counters(s):
    left, right = set(), Counter(s)
    ans = 0
    for step, c in enumerate(s[:-1], 1):
        left.add(c)
        right[c] -= 1
        if right[c] == 0:
            del right[c]
        good = len(left) == len(right)
        ans += good
        print(f"bước {step}: c={c!r} left={sorted(left)} right={dict(right)} "
              f"|L|={len(left)} |R|={len(right)} {'tốt' if good else ''} ans={ans}")
    return ans

print("kết quả:", trace_counters("aacaba"))

bước 1: c='a' left=['a'] right={'a': 3, 'c': 1, 'b': 1} |L|=1 |R|=3  ans=0
bước 2: c='a' left=['a'] right={'a': 2, 'c': 1, 'b': 1} |L|=1 |R|=3  ans=0
bước 3: c='c' left=['a', 'c'] right={'a': 2, 'b': 1} |L|=2 |R|=2 tốt ans=1
bước 4: c='a' left=['a', 'c'] right={'a': 1, 'b': 1} |L|=2 |R|=2 tốt ans=2
bước 5: c='b' left=['a', 'b', 'c'] right={'a': 1} |L|=3 |R|=1  ans=2
kết quả: 2


Trace Cách 4 trên `"acbadbaada"` — in `diff = pre − suf` qua từng vị trí cắt:

In [17]:
def trace_first_last(s):
    n = len(s)
    first, last = {}, {}
    for i, c in enumerate(s):
        first.setdefault(c, i)
        last[c] = i
    print("first =", first, "| last =", last)
    delta = [0] * (n + 1)
    for c in first:
        delta[first[c] + 1] += 1
        delta[last[c] + 1] += 1
    diff = -len(first)
    ans = 0
    for i in range(1, n):
        diff += delta[i]
        ans += diff == 0
        print(f"cắt tại {i}: {s[:i]!r:12} | {s[i:]!r:12} delta={delta[i]} diff={diff:+d} ans={ans}")
    return ans

print("kết quả:", trace_first_last("acbadbaada"))

first = {'a': 0, 'c': 1, 'b': 2, 'd': 4} | last = {'a': 9, 'c': 1, 'b': 5, 'd': 8}
cắt tại 1: 'a'          | 'cbadbaada'  delta=1 diff=-3 ans=0
cắt tại 2: 'ac'         | 'badbaada'   delta=2 diff=-1 ans=0
cắt tại 3: 'acb'        | 'adbaada'    delta=1 diff=+0 ans=1
cắt tại 4: 'acba'       | 'dbaada'     delta=0 diff=+0 ans=2
cắt tại 5: 'acbad'      | 'baada'      delta=1 diff=+1 ans=2
cắt tại 6: 'acbadb'     | 'aada'       delta=1 diff=+2 ans=2
cắt tại 7: 'acbadba'    | 'ada'        delta=0 diff=+2 ans=2
cắt tại 8: 'acbadbaa'   | 'da'         delta=0 diff=+2 ans=2
cắt tại 9: 'acbadbaad'  | 'a'          delta=1 diff=+3 ans=2
kết quả: 2


## 9. Code nộp LeetCode

Bản chính: hai bộ đếm cập nhật dần (Cách 3).

In [18]:
class Solution:
    def numSplits(self, s: str) -> int:
        right = Counter(s)
        left = set()
        ans = 0
        for c in s:
            left.add(c)
            right[c] -= 1
            if right[c] == 0:
                del right[c]
            if len(left) == len(right):
                ans += 1
        return ans

for s, want in examples:
    assert Solution().numSplits(s) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Bất biến (Cách 3):** sau khi xử lý `k` ký tự đầu, `left` = tập chữ cái của `s[:k]` và `right` = bộ đếm (chỉ khoá dương) của `s[k:]`. Đúng lúc đầu (`k = 0`: `left = ∅`, `right = Counter(s)`). Bước chuyển: chữ `c = s[k]` rời phần phải (giảm đếm, xoá khi 0) và vào phần trái (`add`). Vì chỉ giữ khoá dương, `len(right)` = số chữ khác nhau của `s[k:]`. Vậy tại mỗi `k ∈ [1, n−1]` phép so `len(left) == len(right)` kiểm tra đúng điều kiện cắt tốt; `k = n` không bao giờ được tính vì phải rỗng.

**Cách 2:** `pre[i]`, `suf[i]` theo định nghĩa; đếm đúng các `i` hợp lệ.

**Cách 4:** chữ `c` thuộc `s[:i]` ⇔ `first[c] ≤ i − 1`; thuộc `s[i:]` ⇔ `last[c] ≥ i`. Nên `pre[i] − suf[i] = −D + #{c : first[c] + 1 ≤ i} + #{c : last[c] + 1 ≤ i}` (`D` = số chữ khác nhau), chính là tổng tiền tố của `delta`. ∎

## 11. Phân tích Time / Space

`n = len(s) ≤ 10⁵`, bảng chữ cái `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(n)` |
| 2. Tiền tố/hậu tố | `O(n)` | `O(n)` |
| 3. Hai bộ đếm | `O(n)` | `O(Σ)` |
| 4. Đầu/cuối + sai phân | `O(n + Σ)` | `O(n)` |

Thao tác `set`/`Counter` là `O(1)` trung bình (băm); với 26 khoá thực tế luôn rất nhanh.

## 12. Bộ test

Ví dụ đề + ví dụ quen thuộc, edge case (`n = 1`, `n = 2`, toàn một chữ, toàn chữ khác nhau, palindrome), random so với brute (seed cố định), test lớn `n = 10⁵`.

In [19]:
METHODS = {
    "prefix": lambda s: SolutionPrefix().numSplits(s),
    "counter": lambda s: Solution().numSplits(s),
    "counter2": lambda s: SolutionCounter().numSplits(s),
    "firstlast": lambda s: SolutionFirstLast().numSplits(s),
    "firstlast_arr": first_last,
}
edge = [("a", 0), ("ab", 1), ("aa", 1), ("abc", 0), ("abab", 1), ("abba", 1),
        ("abcdefghijklmnopqrstuvwxyz", 1), ("zz" * 5, 9)]
for s, want in examples + edge:
    assert brute(s) == want, s
    for name, f in METHODS.items():
        got = f(s)
        assert got == want, (name, s, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(1525)
for _ in range(600):
    n = rng.randint(1, 30)
    alpha = rng.choice(["ab", "abc", "abcde", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alpha) for _ in range(n))
    ref = brute(s)
    for name, f in METHODS.items():
        assert f(s) == ref, (name, s)
print("600 chuỗi ngẫu nhiên: mọi cách khớp brute")

600 chuỗi ngẫu nhiên: mọi cách khớp brute


In [21]:
rng2 = random.Random(9)
big1 = "".join(rng2.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
big2 = "a" * 50000 + "b" * 50000
for name in ["prefix", "counter", "firstlast"]:
    t = time.perf_counter()
    r1, r2 = METHODS[name](big1), METHODS[name](big2)
    print(f"{name:9s}: {r1} {r2} ({time.perf_counter() - t:.3f}s)")
    assert r2 == 1 and r1 == METHODS["counter"](big1)

prefix   : 99833 1 (0.066s)


counter  : 99833 1 (0.048s)


firstlast: 99833 1 (0.034s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không xoá khoá khi đếm về 0

In [22]:
def wrong_no_delete(s):
    left, right = set(), Counter(s)
    ans = 0
    for c in s[:-1]:
        left.add(c)
        right[c] -= 1                                   # SAI: thiếu del khi = 0
        ans += len(left) == len(right)
    return ans

for s, want in examples:
    print(repr(s), "| sai:", wrong_no_delete(s), "| đúng:", want)

'aacaba' | sai: 1 | đúng: 2
'abcd' | sai: 0 | đúng: 1
'aaaaa' | sai: 4 | đúng: 4
'acbadbaada' | sai: 5 | đúng: 2


### Lỗi 2 — Lệch chỉ số khi ghép tiền tố và hậu tố

Định nghĩa `pre[i]` = chữ khác nhau của `s[0..i]` (**gồm** `i`) và `suf[i]` = của `s[i..]` (**gồm** `i`) rồi so `pre[i] == suf[i]` — ký tự `s[i]` bị tính ở **cả hai** phía.

In [23]:
def wrong_offset(s):
    n = len(s)
    pre, suf = [0] * n, [0] * n
    seen = set()
    for i in range(n):
        seen.add(s[i]); pre[i] = len(seen)
    seen = set()
    for i in range(n - 1, -1, -1):
        seen.add(s[i]); suf[i] = len(seen)
    return sum(1 for i in range(n - 1) if pre[i] == suf[i])   # SAI: phải là suf[i + 1]

for s, want in examples:
    print(repr(s), "| sai:", wrong_offset(s), "| đúng:", want)

'aacaba' | sai: 1 | đúng: 2
'abcd' | sai: 0 | đúng: 1
'aaaaa' | sai: 4 | đúng: 4
'acbadbaada' | sai: 2 | đúng: 2


### Lỗi 3 — So sánh **bộ đếm** thay vì **số chữ khác nhau**

`Counter(left) == Counter(right)` đòi hỏi hai phía có **cùng số lần xuất hiện** của từng chữ — điều kiện mạnh hơn nhiều so với đề.

In [24]:
def wrong_compare_counter(s):
    return sum(1 for i in range(1, len(s)) if Counter(s[:i]) == Counter(s[i:]))   # SAI

for s, want in examples:
    print(repr(s), "| sai:", wrong_compare_counter(s), "| đúng:", want)

'aacaba' | sai: 0 | đúng: 2
'abcd' | sai: 0 | đúng: 1
'aaaaa' | sai: 0 | đúng: 4
'acbadbaada' | sai: 0 | đúng: 2


### Lỗi 4 — So sánh **tập** chữ thay vì **số lượng**

`set(left) == set(right)` yêu cầu hai phía có đúng cùng các chữ; đề chỉ cần **cùng số lượng** (ví dụ `"ab" | "cd"` là tốt).

In [25]:
def wrong_compare_set(s):
    return sum(1 for i in range(1, len(s)) if set(s[:i]) == set(s[i:]))   # SAI

for s, want in examples:
    print(repr(s), "| sai:", wrong_compare_set(s), "| đúng:", want)

'aacaba' | sai: 0 | đúng: 2
'abcd' | sai: 0 | đúng: 1
'aaaaa' | sai: 4 | đúng: 4
'acbadbaada' | sai: 0 | đúng: 2


### Lỗi 5 — Brute force đúng nhưng quá chậm

Tạo `set(s[:i])` mỗi vòng là `O(n²)`; đo trên `n = 10000` để thấy chênh lệch.

In [26]:
s20k = big1[:10000]
t = time.perf_counter(); r_b = brute(s20k); tb = time.perf_counter() - t
t = time.perf_counter(); r_f = Solution().numSplits(s20k); tf = time.perf_counter() - t
print(f"brute: {r_b} ({tb:.2f}s) | O(n): {r_f} ({tf:.4f}s) | nhanh hơn ~{tb / max(tf, 1e-9):.0f} lần")

brute: 9822 (0.87s) | O(n): 9822 (0.0023s) | nhanh hơn ~372 lần


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Cắt mảng/chuỗi thành hai phần, so một đại lượng giữa hai phía"** → mảng tiền tố + mảng hậu tố, hoặc hai cấu trúc cập nhật dần khi dời điểm cắt.
- Đại lượng "số phần tử khác nhau" → `set` (thêm) và `Counter` (bớt, nhớ xoá khoá 0).
- Bảng chữ cái nhỏ → bitmask hoặc mảng 26.
- Mỗi chữ chỉ đổi trạng thái tại **lần đầu** và **lần cuối** → tư duy "sự kiện" + sai phân.
- Các bài cùng khuôn: LC 2256 (Minimum Average Difference), LC 1422 (Max Score After Splitting a String), LC 915 (Partition Array into Disjoint Intervals).

## 15. Biến thể của bài

1. **Các vị trí tốt tạo thành một đoạn liên tiếp** (vì `pre` không giảm, `suf` không tăng) → có thể tìm bằng hai lần binary search trên `pre − suf`.
2. **LC 1422 – Maximum Score After Splitting a String:** đại lượng là số 0 bên trái + số 1 bên phải.
3. **Chia thành 3 phần** có cùng số chữ khác nhau (bài tập 5).
4. **Đếm cắt sao cho |trái| khác nhau ≥ |phải|** → vẫn dùng `pre`, `suf`.
5. **Mảng số nguyên lớn** thay vì chữ cái → `Counter`/`dict` (không dùng được bitmask 26 bit).

In [27]:
# Minh hoạ biến thể 1: các vị trí cắt tốt luôn liên tiếp
def good_positions(s):
    n = len(s)
    return [i for i in range(1, n) if len(set(s[:i])) == len(set(s[i:]))]

rng3 = random.Random(15)
for _ in range(500):
    s = "".join(rng3.choice("abcd") for _ in range(rng3.randint(2, 20)))
    pos = good_positions(s)
    assert pos == list(range(pos[0], pos[-1] + 1)) if pos else True
print("ví dụ:", {s: good_positions(s) for s in ["aacaba", "acbadbaada", "aaaaa"]})
print("Trên 500 chuỗi ngẫu nhiên: vị trí tốt luôn là một đoạn liên tiếp")

ví dụ: {'aacaba': [3, 4], 'acbadbaada': [3, 4], 'aaaaa': [1, 2, 3, 4]}
Trên 500 chuỗi ngẫu nhiên: vị trí tốt luôn là một đoạn liên tiếp


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | `set` hai phía tại mỗi điểm cắt | `O(n²)` | `O(n)` | Đối chiếu |
| 2. Tiền tố/hậu tố | `pre[i] == suf[i]` | `O(n)` | `O(n)` | Dễ hiểu, dễ mở rộng |
| 3. Hai bộ đếm | `set` trái + `Counter` phải | `O(n)` | `O(Σ)` | Mặc định, gợi ý của đề |
| 4. Đầu/cuối + sai phân | `diff = pre − suf` qua các sự kiện | `O(n + Σ)` | `O(n)` | Tư duy sự kiện, nâng cao |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numSplits("aaaa")`, `("abcd")`, `("ab")`, `("abcba")` trả về gì?

### Bài 2 (tính tay)
Với `s = "acbadbaada"`, lập bảng `pre[i]`, `suf[i]` cho `i = 1..9` và chỉ ra các vị trí cắt tốt.

### Bài 3 (điền TODO)
Viết `distinct_prefix(s)`: list `pre` độ dài `n + 1` với `pre[i]` = số chữ khác nhau của `s[:i]`, dùng **bitmask**.

### Bài 4 (tự viết lại)
Viết `count_left_more(s)`: số vị trí cắt (hai phía khác rỗng) mà số chữ khác nhau bên **trái lớn hơn** bên phải.

### Bài 5 (mini challenge)
Viết `three_way_splits(s)`: số cách chia `s` thành **3 phần khác rỗng** có cùng số chữ khác nhau.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def distinct_prefix(s):
    # TODO: mask |= 1 << (ord(c) - 97); pre.append(bin(mask).count("1"))
    pass

# Bài 4 — TODO
def count_left_more(s):
    # TODO: như Cách 3 nhưng đếm len(left) > len(right) với i trong [1, n-1]
    pass

# Bài 5 — TODO
def three_way_splits(s):
    # TODO: pre, suf như Cách 2; với mỗi i (cuối phần 1), duyệt j tăng dần,
    #       giữ set phần giữa s[i:j]; đếm khi pre[i] == len(mid) == suf[j]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"aaaa"`: mọi cách cắt đều 1 vs 1. (2) `"abcd"`: chỉ cắt giữa. (3) `3, 1, 1, ?` — với `"abcba"` thử từng vị trí: `a|bcba` 1 vs 3, `ab|cba` 2 vs 3, `abc|ba` 3 vs 2, `abcb|a` 3 vs 1 → 0.

**Bài 2.** (1) Dùng code `pre/suf`. (2) `pre` tăng tại lần đầu của `a, c, b, d`. (3) `suf` giảm sau lần cuối của mỗi chữ; hai vị trí tốt.

**Bài 3.** (1) `pre = [0]`. (2) Mỗi ký tự OR bit tương ứng. (3) Đếm bit bằng `bin(mask).count("1")`.

**Bài 4.** (1) Giữ nguyên khung Cách 3. (2) Chỉ xét `c in s[:-1]`. (3) Điều kiện `>`.

**Bài 5.** (1) `pre[i]` cho phần 1 = `s[:i]`, `suf[j]` cho phần 3 = `s[j:]`, `1 ≤ i < j ≤ n − 1`. (2) Cố định `i`, tăng `j` và thêm `s[j−1]` vào set giữa. (3) `O(n²)` là đủ cho bài tập; đối chiếu brute `O(n³)`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
r1 = [Solution().numSplits(x) for x in ["aaaa", "abcd", "ab", "abcba"]]
print("Bài 1:", r1)
assert r1 == [3, 1, 1, 0]

# Bài 2
s = "acbadbaada"
n = len(s)
rows = [(i, len(set(s[:i])), len(set(s[i:]))) for i in range(1, n)]
print("Bài 2 (i, pre, suf):", rows)
assert [i for i, a, b in rows if a == b] == [3, 4]

Bài 1: [3, 1, 1, 0]
Bài 2 (i, pre, suf): [(1, 1, 4), (2, 2, 3), (3, 3, 3), (4, 3, 3), (5, 4, 3), (6, 4, 2), (7, 4, 2), (8, 4, 2), (9, 4, 1)]


In [30]:
# Bài 3
def distinct_prefix_sol(s):
    pre, mask = [0], 0
    for c in s:
        mask |= 1 << (ord(c) - 97)
        pre.append(bin(mask).count("1"))
    return pre

# Bài 4
def count_left_more_sol(s):
    left, right = set(), Counter(s)
    ans = 0
    for c in s[:-1]:
        left.add(c)
        right[c] -= 1
        if right[c] == 0:
            del right[c]
        ans += len(left) > len(right)
    return ans

rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("abcdef") for _ in range(rng.randint(1, 25)))
    assert distinct_prefix_sol(s) == [len(set(s[:i])) for i in range(len(s) + 1)]
    assert count_left_more_sol(s) == sum(1 for i in range(1, len(s)) if len(set(s[:i])) > len(set(s[i:])))
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
# Bài 5
def three_way_splits_sol(s):
    n = len(s)
    pre = distinct_prefix_sol(s)
    suf = [0] * (n + 1)
    seen = set()
    for i in range(n - 1, -1, -1):
        seen.add(s[i]); suf[i] = len(seen)
    ans = 0
    for i in range(1, n - 1):
        mid = set()
        for j in range(i + 1, n):
            mid.add(s[j - 1])
            if pre[i] == len(mid) == suf[j]:
                ans += 1
    return ans

def brute_three(s):
    n = len(s)
    return sum(1 for i in range(1, n) for j in range(i + 1, n)
               if len(set(s[:i])) == len(set(s[i:j])) == len(set(s[j:])))

print("Bài 5:", three_way_splits_sol("aaaa"), three_way_splits_sol("abcabc"), three_way_splits_sol("abc"))
for _ in range(300):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 18)))
    assert three_way_splits_sol(s) == brute_three(s)
print("Đáp án Bài 1–5: mọi assert đều qua")

Bài 5: 3 1 1
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "dời điểm cắt, cập nhật hai phía":**

```python
right = Counter(s)
left = set()
ans = 0
for c in s[:-1]:                # hai phía khác rỗng
    left.add(c)
    right[c] -= 1
    if right[c] == 0:
        del right[c]            # để len(right) = số chữ khác nhau
    ans += len(left) == len(right)
```

**Checklist trước khi submit:**
- [ ] Xoá khoá khỏi `Counter` khi về 0 (hoặc tự đếm số khoá dương).
- [ ] Ghép đúng `pre[i]` (tiền tố độ dài `i`) với `suf[i]` (hậu tố bắt đầu tại `i`).
- [ ] So **số lượng** chữ khác nhau, không so tập hay bộ đếm.
- [ ] Vị trí cắt `1 ≤ i ≤ n − 1`.
- [ ] Đúng chữ ký: `numSplits(self, s: str) -> int`.